# Naive Bayes Classification — ML Algorithms Lab

## Cheat-sheet items checked carefully
The supplied cheat sheet explicitly lists these Naive Bayes model classes:
- **`GaussianNB()`** — note in the sheet: *no scaling needed*.
- **`MultinomialNB()`** — note in the sheet: *non-negative inputs only*.
- **`BernoulliNB()`** — listed as a model option.

**Important source limitation:** the cheat sheet does **not** provide a Naive Bayes parameter grid or numeric tuning values. Therefore, the notebook below clearly separates:
1. the three model variants explicitly named in the cheat sheet; and
2. practical candidate values added for GridSearchCV (these are additional choices, not values quoted from the sheet).

The notebook includes preprocessing appropriate to each estimator, stratified cross-validation, GridSearchCV, multiple metrics, plots, paired statistical tests, and CSV outputs. Run from top to bottom and enter your CSV path and target column when prompted. Syntax is checked; results depend on your dataset and have not been run on it.


In [ ]:
# =========================
# 1. Imports
# =========================
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, wilcoxon, friedmanchisquare
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, Binarizer, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate, learning_curve
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, classification_report, ConfusionMatrixDisplay,
    log_loss, roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.base import clone
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports complete. random_state =", RANDOM_STATE)

# =========================
# 2. Load CSV
# =========================
csv_files = glob.glob("*.csv")
if len(csv_files) == 1:
    default_path = csv_files[0]
    print("Detected CSV:", default_path)
    path = input("Press Enter to use it, or enter another CSV path: ").strip() or default_path
elif len(csv_files) > 1:
    print("CSV files:", csv_files)
    path = input("Enter CSV path: ").strip()
else:
    path = input("Enter CSV path: ").strip()
if not os.path.exists(path):
    raise FileNotFoundError(f"File not found: {path}")
df = pd.read_csv(path)
print("Shape:", df.shape)
display(df.head())
print("\\nData types:")
df.info()
print("\\nSummary:")
display(df.describe(include="all").T)
print("\\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(30))
print("Duplicate rows:", df.duplicated().sum())

# =========================
# 3. Target selection, cleaning, EDA
# =========================
target = input("Enter exact target/class column name: ").strip()
if target not in df.columns:
    raise ValueError(f"Target not found. Available columns: {list(df.columns)}")
df = df.drop_duplicates().dropna(subset=[target]).copy()
y = df[target].copy()
X = df.drop(columns=[target]).copy()

id_like = [c for c in X.columns if (
    c.lower() in {"id", "index", "row_id", "customerid", "customer_id", "record_id"} or
    (X[c].nunique(dropna=True) >= 0.98 * max(len(X), 1) and
     (pd.api.types.is_integer_dtype(X[c]) or X[c].dtype == "object"))
)]
if id_like:
    print("Potential identifier columns excluded (please review):", id_like)
    X = X.drop(columns=id_like)
if y.nunique() < 2:
    raise ValueError("Classification needs at least two target classes.")
print("Features:", X.shape, "| Classes:", y.nunique())
display(y.value_counts().rename("class_count").to_frame())

plt.figure(figsize=(8,4))
y.astype(str).value_counts().plot(kind="bar")
plt.title("Target class distribution"); plt.xlabel("Class"); plt.ylabel("Count")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
if num_cols:
    X[num_cols].hist(figsize=(12, max(4, 2.5*((len(num_cols)+3)//4))), bins=20)
    plt.suptitle("Numeric feature distributions"); plt.tight_layout(); plt.show()
    if len(num_cols) > 1:
        corr = X[num_cols].corr(numeric_only=True)
        plt.figure(figsize=(9,6)); plt.imshow(corr, aspect="auto"); plt.colorbar(label="Correlation")
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title("Numeric feature correlation"); plt.tight_layout(); plt.show()
for col in cat_cols[:5]:
    plt.figure(figsize=(8,3))
    X[col].astype(str).value_counts().head(12).plot(kind="bar")
    plt.title(f"Most frequent categories: {col}"); plt.tight_layout(); plt.show()

# =========================
# 4. Split before fitting any transformations
# =========================
if int(y.value_counts().min()) < 2:
    raise ValueError("At least one class has fewer than two examples; stratified CV cannot be performed.")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
min_train_class = int(y_train.value_counts().min())
n_splits = min(5, min_train_class)
if n_splits < 2:
    raise ValueError("Insufficient examples per class in training set for CV.")
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape, "| CV folds:", n_splits)

# =========================
# 5. Preprocessing pipelines
# =========================
# One-hot encoded categorical variables are non-negative.
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

# GaussianNB: sheet says no scaling is needed; median-impute numeric columns,
# one-hot encode categoricals, and use a dense matrix.
gaussian_num = Pipeline([("imputer", SimpleImputer(strategy="median"))])
gaussian_cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)])
gaussian_transformers = []
if num_cols: gaussian_transformers.append(("num", gaussian_num, num_cols))
if cat_cols: gaussian_transformers.append(("cat", gaussian_cat, cat_cols))
prep_gaussian = ColumnTransformer(gaussian_transformers, remainder="drop", sparse_threshold=0)

# MultinomialNB requires non-negative values: MinMax-scale numeric data to [0,1],
# impute categories and one-hot encode them.
multi_num = Pipeline([("imputer", SimpleImputer(strategy="median")), ("minmax", MinMaxScaler())])
multi_cat = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)])
multi_transformers = []
if num_cols: multi_transformers.append(("num", multi_num, num_cols))
if cat_cols: multi_transformers.append(("cat", multi_cat, cat_cols))
prep_multinomial = ColumnTransformer(multi_transformers, remainder="drop", sparse_threshold=0)

# BernoulliNB models binary features; binarize the processed feature matrix at 0.5.
prep_bernoulli = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("minmax", MinMaxScaler())]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)]), cat_cols)
    ], remainder="drop", sparse_threshold=0)),
    ("binarize", Binarizer(threshold=0.5))
])

# =========================
# 6. Model variants and parameter grids
# =========================
# SOURCE-DERIVED: GaussianNB, MultinomialNB, BernoulliNB are explicitly listed in the cheat sheet.
# SOURCE-DERIVED constraints: GaussianNB does not need scaling; MultinomialNB needs non-negative inputs.
# ADDITIONAL PRACTICAL GRID VALUES (not specified by the cheat sheet):
# GaussianNB: var_smoothing; MultinomialNB/BernoulliNB: alpha and fit_prior.
models = {
    "GaussianNB": {
        "pipeline": Pipeline([("prep", prep_gaussian), ("model", GaussianNB())]),
        "grid": {"model__var_smoothing": [1e-11, 1e-9, 1e-7, 1e-5]}
    },
    "MultinomialNB": {
        "pipeline": Pipeline([("prep", prep_multinomial), ("model", MultinomialNB())]),
        "grid": {"model__alpha": [0.1, 0.5, 1.0, 2.0], "model__fit_prior": [True, False]}
    },
    "BernoulliNB": {
        "pipeline": Pipeline([("prep", prep_bernoulli), ("model", BernoulliNB())]),
        "grid": {"model__alpha": [0.1, 0.5, 1.0, 2.0], "model__fit_prior": [True, False]}
    }
}
print("Models in this notebook:", list(models))
print("The numeric grid values are practical additions; the cheat sheet gives no Naive Bayes grid values.")

# =========================
# 7. Metrics helper
# =========================
def calculate_metrics(y_true, y_pred, y_prob=None, classes=None):
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "matthews_corrcoef": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred)
    }
    if y_prob is not None:
        try:
            if len(classes) == 2:
                out["roc_auc"] = roc_auc_score(y_true, y_prob[:, 1])
                out["average_precision"] = average_precision_score(y_true, y_prob[:, 1])
            else:
                out["roc_auc_ovr_macro"] = roc_auc_score(
                    y_true, y_prob, labels=classes, multi_class="ovr", average="macro")
                out["average_precision_macro"] = average_precision_score(
                    label_binarize(y_true, classes=classes), y_prob, average="macro")
            out["log_loss"] = log_loss(y_true, y_prob, labels=classes)
        except Exception:
            pass
    return out

# =========================
# 8. GridSearchCV + cross-validated metrics for all three variants
# =========================
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted"
}
searches, cv_rows, time_rows = {}, [], []
for name, spec in models.items():
    print("\\n" + "="*70 + "\\nTuning:", name)
    search = GridSearchCV(
        estimator=spec["pipeline"], param_grid=spec["grid"],
        scoring=scoring, refit="f1_macro", cv=cv, n_jobs=-1,
        error_score=np.nan, return_train_score=True
    )
    t0 = time.time()
    search.fit(X_train, y_train)
    elapsed = time.time() - t0
    searches[name] = search
    idx = search.best_index_
    row = {
        "model": name,
        "best_params": str(search.best_params_),
        "best_cv_macro_f1": search.best_score_,
        "cv_accuracy": search.cv_results_["mean_test_accuracy"][idx],
        "cv_balanced_accuracy": search.cv_results_["mean_test_balanced_accuracy"][idx],
        "cv_precision_macro": search.cv_results_["mean_test_precision_macro"][idx],
        "cv_recall_macro": search.cv_results_["mean_test_recall_macro"][idx],
        "cv_f1_weighted": search.cv_results_["mean_test_f1_weighted"][idx],
        "fit_seconds": elapsed
    }
    cv_rows.append(row)
    print("Best parameters:", search.best_params_)
    print("Best CV macro-F1:", round(search.best_score_, 5))
    print(f"Search time: {elapsed:.1f} seconds")
    display(pd.DataFrame(search.cv_results_).sort_values("rank_test_f1_macro")[
        ["rank_test_f1_macro", "mean_test_f1_macro", "std_test_f1_macro", "mean_test_accuracy", "params"]
    ].head(10))

cv_summary = pd.DataFrame(cv_rows).sort_values("best_cv_macro_f1", ascending=False)
print("\\nCross-validation comparison:")
display(cv_summary)

plt.figure(figsize=(8,4))
plt.bar(cv_summary["model"], cv_summary["best_cv_macro_f1"])
plt.ylabel("Best CV macro-F1"); plt.title("Naive Bayes variants — CV comparison")
plt.tight_layout(); plt.show()

# =========================
# 9. Held-out test metrics and visualizations
# =========================
test_rows = []
best_estimators = {}
for name, search in searches.items():
    model = search.best_estimator_
    best_estimators[name] = model
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)
    classes = list(model.named_steps["model"].classes_)
    metrics = calculate_metrics(y_test, pred, prob, classes)
    test_rows.append({"model": name, **metrics})
    print("\\n", "="*65, "\\n", name, "\\nBest parameters:", search.best_params_)
    display(pd.Series(metrics, name="test_score").to_frame())
    print(classification_report(y_test, pred, zero_division=0))
    fig, ax = plt.subplots(figsize=(6,5))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, xticks_rotation=45)
    ax.set_title(f"{name} — confusion matrix")
    plt.tight_layout(); plt.show()
    try:
        if len(classes) == 2:
            RocCurveDisplay.from_predictions(y_test, prob[:, 1])
            plt.title(f"{name} — ROC curve"); plt.tight_layout(); plt.show()
            PrecisionRecallDisplay.from_predictions(y_test, prob[:, 1])
            plt.title(f"{name} — precision-recall curve"); plt.tight_layout(); plt.show()
    except Exception as e:
        print("Binary ROC/PR plotting skipped:", e)

test_summary = pd.DataFrame(test_rows).sort_values("f1_macro", ascending=False)
print("\\nHeld-out test-set comparison:")
display(test_summary)
plt.figure(figsize=(8,4))
plt.bar(test_summary["model"], test_summary["f1_macro"])
plt.ylabel("Test macro-F1"); plt.title("Naive Bayes variants — held-out test comparison")
plt.tight_layout(); plt.show()

# =========================
# 10. Paired significance tests across the same CV folds
# =========================
# Re-score each tuned estimator on identical folds. Tuning above is done on the training split;
# this fold comparison is descriptive and the CV folds are not independent observations.
fold_scores = {}
for name, model in best_estimators.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring="f1_macro",
                            n_jobs=-1, return_train_score=False)
    fold_scores[name] = scores["test_score"]
fold_table = pd.DataFrame(fold_scores)
print("Matched-fold macro-F1 scores:")
display(fold_table)
names = list(fold_scores.keys())
if len(names) >= 3:
    try:
        print("Friedman test across variants:", friedmanchisquare(*(fold_scores[n] for n in names)))
    except Exception as e:
        print("Friedman test unavailable:", e)
for i in range(len(names)):
    for j in range(i+1, len(names)):
        a, b = fold_scores[names[i]], fold_scores[names[j]]
        print(f"\\nPaired comparison: {names[i]} vs {names[j]}")
        print("Mean macro-F1 difference:", round(float(np.mean(a-b)), 6))
        try: print("Paired t-test:", ttest_rel(a, b, nan_policy="omit"))
        except Exception as e: print("Paired t-test unavailable:", e)
        try: print("Wilcoxon signed-rank:", wilcoxon(a, b, zero_method="wilcox"))
        except Exception as e: print("Wilcoxon unavailable:", e)
print("Interpret significance tests cautiously: a small number of CV folds gives low statistical power.")

# =========================
# 11. Learning curve for best-performing CV variant
# =========================
winner_name = cv_summary.iloc[0]["model"]
winner = best_estimators[winner_name]
print("Top CV variant:", winner_name)
try:
    lc_splits = min(3, int(y_train.value_counts().min()))
    if lc_splits >= 2:
        lc_cv = StratifiedKFold(n_splits=lc_splits, shuffle=True, random_state=RANDOM_STATE)
        sizes, train_scores, val_scores = learning_curve(
            winner, X_train, y_train, cv=lc_cv, scoring="f1_macro",
            train_sizes=np.linspace(0.3, 1.0, 4), n_jobs=-1, error_score=np.nan
        )
        plt.figure(figsize=(8,5))
        plt.plot(sizes, np.nanmean(train_scores, axis=1), marker="o", label="Train macro-F1")
        plt.plot(sizes, np.nanmean(val_scores, axis=1), marker="o", label="Validation macro-F1")
        plt.xlabel("Training examples"); plt.ylabel("Macro-F1")
        plt.title(f"Learning curve — {winner_name}"); plt.legend()
        plt.tight_layout(); plt.show()
except Exception as e:
    print("Learning curve skipped:", e)

# =========================
# 12. Export summaries and predictions
# =========================
cv_summary.to_csv("naive_bayes_cv_summary.csv", index=False)
test_summary.to_csv("naive_bayes_test_summary.csv", index=False)
for name, model in best_estimators.items():
    pred = model.predict(X_test)
    safe_name = name.lower().replace(" ", "_")
    pd.DataFrame({
        "actual": y_test.reset_index(drop=True),
        "predicted": pd.Series(pred).reset_index(drop=True)
    }).to_csv(f"{safe_name}_test_predictions.csv", index=False)
print("Saved CV/test summaries and one prediction CSV per Naive Bayes variant.")
print("Notebook complete.")
